#  Smart City Traffic – Model Training & Evaluation

Trains and compares:
- Linear Regression (baseline)
- Random Forest Regressor
- Gradient Boosting Regressor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, sys, os
sys.path.insert(0, '../src')
warnings_off = __import__('warnings').filterwarnings('ignore')

from data_preprocessing  import load_data, clean_data
from feature_engineering import extract_features, get_feature_columns
from train_model         import run_training
from evaluate_model      import run_evaluation, compute_metrics

from sklearn.model_selection import train_test_split
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='darkgrid')
print('Ready ')

Ready ✅


## 1. Prepare Data

In [2]:
df_raw   = load_data('../data/raw/traffic_data.csv')
df_clean = clean_data(df_raw)
df_feat  = extract_features(df_clean)

feat_cols = get_feature_columns()
feat_cols = [c for c in feat_cols if c in df_feat.columns]

X = df_feat[feat_cols].values
y = df_feat['Vehicles'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

✅ Loaded data: 48,204 rows × 8 columns
No DateTime column found. Creating synthetic DateTime.
  Dropped 0 duplicate rows.
✅ Cleaned data: 48,204 rows remaining.
 Features extracted. Total columns: 28
Train: (38563, 18), Test: (9641, 18)


## 2. Train & Compare Models

In [3]:
from sklearn.linear_model  import LinearRegression
from sklearn.ensemble      import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline      import Pipeline

models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=12, n_jobs=-1, random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42
    ),
}

results = []
for name, m in models.items():
    print(f'Training {name}...')
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    results.append({
        'Model': name,
        'RMSE':  np.sqrt(mean_squared_error(y_test, preds)),
        'MAE':   mean_absolute_error(y_test, preds),
        'R²':    r2_score(y_test, preds),
    })

results_df = pd.DataFrame(results).set_index('Model')
print('\n', results_df.round(4))

Training Linear Regression...
Training Random Forest...
Training Gradient Boosting...

                         RMSE        MAE      R²
Model                                          
Linear Regression  1986.0542  1745.4353  0.0023
Random Forest      1510.5554  1262.4797  0.4228
Gradient Boosting  1863.8499  1613.2529  0.1213


## 3. Visualise Model Comparison

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, ['RMSE', 'MAE', 'R²']):
    results_df[metric].plot(
        kind='bar', ax=ax, color=['#0077B6','#48CAE4','#90E0EF'], edgecolor='white'
    )
    ax.set_title(metric, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('Model Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/graphs/model_comparison.png', dpi=150)
plt.show()

## 4. Best Model – Predictions

In [5]:
# Use Random Forest (typically best performer)
best = models['Random Forest']
preds = best.predict(X_test)

n = 300
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y_test[:n],  label='Actual',    color='#0077B6', lw=1.5)
ax.plot(preds[:n],   label='Predicted', color='#90E0EF', lw=1.5, alpha=0.85)
ax.set_title('Random Forest – Actual vs Predicted (first 300 test samples)', fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Vehicles')
ax.legend()
plt.tight_layout()
plt.savefig('../results/graphs/rf_actual_vs_predicted.png', dpi=150)
plt.show()

## 5. Feature Importances

In [6]:
importances = best.feature_importances_
idx = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(idx)), importances[idx], color='#0077B6', edgecolor='white')
ax.set_xticks(range(len(idx)))
ax.set_xticklabels([feat_cols[i] for i in idx], rotation=40, ha='right', fontsize=9)
ax.set_title('Top 15 Feature Importances – Random Forest', fontweight='bold')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('../results/graphs/feature_importance_nb.png', dpi=150)
plt.show()

## 6. Save Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(best,      '../models/traffic_model.pkl')
joblib.dump(feat_cols, '../models/feature_cols.pkl')
print('Model saved ')

Model saved ✅
